<a href="https://colab.research.google.com/github/Luisfegit/Data-en-concierto/blob/main/data_en_concierto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/Luisfegit/Data-en-concierto/refs/heads/main/spotify_data%20clean.csv"

raw = pd.read_csv(URL)

print("Dimensiones del dataset:", raw.shape)
display(raw.head())


Dimensiones del dataset: (8582, 15)


,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


In [ ]:
# ============================================================
# 4. CARGA E INSPECCIÓN INICIAL - Dataset Spotify
# Pega cada bloque "# %%" en una celda distinta de Google Colab
# ============================================================

# %% [1] Librerías
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)


# %% [2] Cargar el archivo
# OPCIÓN A: subir el archivo directamente a Colab (la más simple)
from google.colab import files
subido = files.upload()                      # selecciona dataset_spotify_sp.xlsx
RUTA = list(subido.keys())[0]

# OPCIÓN B: desde Google Drive (comenta la opción A y descomenta estas líneas)
# from google.colab import drive
# drive.mount("/content/drive")
# RUTA = "/content/drive/MyDrive/ruta/a/dataset_spotify_sp.xlsx"

# La hoja con datos se llama "track_data_final" (la otra, "Hoja1", está vacía).
# dtype=str en track_name evita que títulos como "3" se lean como número.
raw = pd.read_excel(RUTA, sheet_name="track_data_final", dtype={"track_name": str})

# df = copia de trabajo; raw queda intacto como respaldo
df = raw.copy()


# %% [3] Tamaño de la base
print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")


# %% [4] Muestra pequeña de filas
print("Primeras 5 filas:")
display(df.head(5))

print("\n5 filas al azar:")
display(df.sample(5, random_state=42))


# %% [5] Nombres y tipos de variables (+ nulos y valores únicos)
info = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(),
})
display(info)


# %% [6] Diccionario breve de variables
# Significado y rol son conocidos de antemano; tipo, nulos y únicos se calculan.
significado = {
    "track_id":           ("Identificador único de la canción en Spotify", "identificador"),
    "track_name":         ("Nombre de la canción", "texto"),
    "track_number":       ("Posición de la canción dentro de su álbum", "numérica"),
    "track_popularity":   ("Popularidad de la canción (0 a 100)", "resultado de interés"),
    "track_duration_ms":  ("Duración en milisegundos", "numérica"),
    "explicit":           ("Si la canción tiene contenido explícito", "categórica (booleana)"),
    "artist_name":        ("Nombre del artista principal", "categórica (alta cardinalidad)"),
    "artist_popularity":  ("Popularidad del artista (escala 0 a 1000)", "numérica / posible resultado"),
    "artist_followers":   ("Seguidores del artista en Spotify", "numérica"),
    "artist_genres":      ("Lista de géneros del artista, guardada como texto", "categórica (lista)"),
    "album_id":           ("Identificador único del álbum", "identificador"),
    "album_name":         ("Nombre del álbum", "texto"),
    "album_release_date": ("Fecha de lanzamiento del álbum", "fecha"),
    "album_total_tracks": ("Total de canciones del álbum", "numérica"),
    "album_type":         ("Tipo de lanzamiento: album, single o compilation", "categórica"),
}

def observaciones(col):
    """Observaciones calculadas a partir de los datos reales."""
    s = df[col]
    obs = []
    if s.isna().any():
        obs.append(f"{s.isna().sum()} nulos")
    if col == "artist_popularity":
        obs.append(f"rango {s.min():.0f}-{s.max():.0f}: escala x10 respecto a Spotify (0-100)")
    if col == "track_popularity":
        obs.append(f"{(s == 0).sum()} canciones con popularidad 0")
    if col == "track_duration_ms":
        obs.append(f"{(s == 0).sum()} con duración 0; promedio {s.mean()/60000:.1f} min")
    if col == "artist_genres":
        obs.append(f"{(s == '[]').sum()} listas vacías ({(s == '[]').mean()*100:.0f} %); hay que convertir a lista")
    if col == "album_release_date":
        obs.append(f"{(s.astype(str).str.len() == 4).sum()} fechas solo con año; convertir con to_datetime")
    if col == "explicit":
        obs.append(f"{s.mean()*100:.0f} % son explícitas")
    if col in ("track_id", "album_id"):
        obs.append("único por canción" if s.is_unique else "se repite (varias canciones por álbum)")
    return "; ".join(obs) if obs else "-"

diccionario = pd.DataFrame({
    "variable":      df.columns,
    "significado":   [significado[c][0] for c in df.columns],
    "tipo":          [str(df[c].dtype) for c in df.columns],
    "rol":           [significado[c][1] for c in df.columns],
    "observaciones": [observaciones(c) for c in df.columns],
})
display(diccionario)


# %% [7] Clasificación de columnas por rol
identificadores = ["track_id", "album_id"]
fechas          = ["album_release_date"]
categoricas     = ["explicit", "artist_name", "artist_genres", "album_type"]
numericas       = ["track_number", "track_popularity", "track_duration_ms",
                   "artist_popularity", "artist_followers", "album_total_tracks"]
texto_libre     = ["track_name", "album_name"]
resultados      = ["track_popularity"]   # variable objetivo principal; artist_popularity como alternativa

print("Identificadores:     ", identificadores)
print("Fechas:              ", fechas)
print("Categóricas:         ", categoricas)
print("Numéricas:           ", numericas)
print("Texto libre:         ", texto_libre)
print("Posible resultado:   ", resultados)

# Verificación rápida de que no se omitió ninguna columna
clasificadas = set(identificadores + fechas + categoricas + numericas + texto_libre)
print("\nColumnas sin clasificar:", set(df.columns) - clasificadas or "ninguna")


# %% [8] Resumen estadístico rápido
print("Variables numéricas:")
display(df[numericas].describe().round(2))

print("\nVariables categóricas (top 5 valores):")
for c in ["explicit", "album_type", "artist_name", "artist_genres"]:
    print(f"\n{c}:")
    display(df[c].value_counts().head(5).to_frame("conteo"))